# HAGI — Cloud Training Launcher

Runs the HAGI Stage 0 pipeline on a cloud GPU. Works on **Google Colab**, 
**Lightning AI**, and **Kaggle** (Jupyter). It auto-detects the GPU and picks the 
right config (bf16 `baseline` on Ampere+, fp16 `colab_t4` on a T4).

Run the cells top to bottom. If a free session is killed, just re-run the **Train** 
cell — `--resume auto` continues from the latest checkpoint.


## 1. Clone the repo


In [ ]:
import os, subprocess
# Clone the active branch — all current work lives on `experimental`, not main.
if not os.path.isdir('HAGI') and os.path.basename(os.getcwd()) != 'HAGI':
    subprocess.run(['git','clone','-b','experimental','https://github.com/ShmidtS/HAGI.git'], check=True)
if os.path.basename(os.getcwd()) != 'HAGI':
    os.chdir('HAGI')
print('cwd:', os.getcwd())

## 2. Install dependencies
Full requirements (tokenization needs `datasets`+`transformers`). A few minutes the first time.


In [ ]:
!pip install -q -r requirements.txt

## 3. Detect the GPU and pick a config
Ampere or newer (sm80+, e.g. A100/L4/L40S/A10G) -> `baseline.yaml` (bf16, full 4096 seq). 
Turing T4 (sm75) -> `colab_t4.yaml` (fp16, seq 1024). On Colab free you cannot choose the 
GPU; this picks correctly for whatever you get.


In [ ]:
import torch
CONFIG = 'configs/colab_t4.yaml'
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    sm = p.major*10 + p.minor
    print(f'GPU: {p.name} | sm{sm} | {p.total_memory/1e9:.0f} GB')
    CONFIG = 'configs/baseline.yaml' if sm >= 80 else 'configs/colab_t4.yaml'
else:
    print('No GPU detected - CPU is validation only, not real training.')
print('chosen config:', CONFIG)

## 4. Pick a persistent checkpoint directory
**Free sessions are wiped on reset.** Colab -> mount Google Drive. Lightning AI -> its 
storage is already persistent. Kaggle -> use `/kaggle/working` and snapshot to a Dataset.


In [ ]:
import os
CKPT_DIR = 'checkpoints'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/hagi'
    print('Colab detected -> checkpoints on Drive:', CKPT_DIR)
except Exception:
    print('Not Colab -> checkpoints at:', CKPT_DIR, '(ensure this is persistent storage)')
os.makedirs(CKPT_DIR, exist_ok=True)

## 5. Tokenize the corpus
Starts with a **20k-document smoke** (fast, ~minutes) so you can confirm the full path 
before committing hours. For the real run, remove `--limit 20000` (re-tokenizes the full 
`sample-10BT`). Tokenize once, then reuse the shards across sessions.


In [ ]:
!python -m prototype.data.tokenize \
    --dataset HuggingFaceFW/fineweb-edu --subset sample-10BT \
    --output data/fineweb-edu --tokenizer HuggingFaceTB/SmolLM2-135M \
    --limit 20000

## 6. Train (auto-resume)
`max_steps` is derived from the config's `train_tokens`. Re-run this exact cell after a 
session dies - it resumes from the latest checkpoint in your persistent dir.


In [ ]:
!python -m prototype.training.train \
    --config {CONFIG} --data data/fineweb-edu \
    --ckpt-dir {CKPT_DIR} --resume auto

## 7. (Optional) Evaluate a checkpoint
Once a checkpoint exists, run standard benchmarks via the lm-eval adapter. 
Needs `pip install lm-eval` (already in requirements).


In [ ]:
# Replace with your latest checkpoint path under CKPT_DIR.
!python -m prototype.evaluation.evaluate \
    --ckpt {CKPT_DIR}/baseline/step-00002000.pt \
    --benchmarks arc_challenge,boolq \
    --device cuda